In [37]:
import pandas as pd
import numpy as np
import os

In [38]:
raw_path = r"C:\Users\Akash Ghosal\Desktop\Essentials\MS Prep\Research Paper\LLM Alpha Research\data\raw\motley-fool-data.pkl"
data = pd.read_pickle(raw_path)
print('Raw Shape', data.shape)
print(data.head())

Raw Shape (18755, 5)
                         date      exchange        q ticker  \
0  Aug 27, 2020, 9:00 p.m. ET  NASDAQ: BILI  2020-Q2   BILI   
1  Jul 30, 2020, 4:30 p.m. ET     NYSE: GFF  2020-Q3    GFF   
2  Oct 23, 2019, 5:00 p.m. ET  NASDAQ: LRCX  2020-Q1   LRCX   
3  Nov 6, 2019, 12:00 p.m. ET  NASDAQ: BBSI  2019-Q3   BBSI   
4   Aug 7, 2019, 8:30 a.m. ET  NASDAQ: CSTE  2019-Q2   CSTE   

                                          transcript  
0  Prepared Remarks:\nOperator\nGood day, and wel...  
1  Prepared Remarks:\nOperator\nThank you for sta...  
2  Prepared Remarks:\nOperator\nGood day and welc...  
3  Prepared Remarks:\nOperator\nGood day, everyon...  
4  Prepared Remarks:\nOperator\nGreetings and wel...  


In [39]:
# Remove timezone text and fix AM/PM formatting
data['date_clean'] = (
    data['date']
    .str.replace(' ET', '', regex=False)
    .str.replace('a.m.', 'AM', regex=False)
    .str.replace('p.m.', 'PM', regex=False)
)

# Convert to datetime
data['date_clean'] = pd.to_datetime(
    data['date_clean'],
    format='%b %d, %Y, %I:%M %p',
    errors='coerce'
)

# IMPORTANT STEP — keep only calendar date (drop time)
data['date_clean'] = data['date_clean'].dt.normalize()

print(data[['date', 'date_clean']].head())


                         date date_clean
0  Aug 27, 2020, 9:00 p.m. ET 2020-08-27
1  Jul 30, 2020, 4:30 p.m. ET 2020-07-30
2  Oct 23, 2019, 5:00 p.m. ET 2019-10-23
3  Nov 6, 2019, 12:00 p.m. ET 2019-11-06
4   Aug 7, 2019, 8:30 a.m. ET 2019-08-07


In [40]:
print('missing dates:', data['date_clean'].isna().sum())

missing dates: 742


In [41]:
data = data.dropna(subset=['date_clean'])
data = data.dropna(subset=['transcript'])
print("Shape after removing nulls:", data.shape)

Shape after removing nulls: (18013, 6)


In [42]:
# Remove Short Transcripts
data['transcript_length'] = data['transcript'].str.len()
data = data[data['transcript_length'] > 500]
print("Shape after length filter:", data.shape)

Shape after length filter: (18013, 7)


In [43]:
clean_data = data[['ticker', 'date_clean', 'transcript']].copy()
clean_data = clean_data.rename(columns={'date_clean': 'date'})
print(clean_data.head())

clean_data.shape

  ticker       date                                         transcript
0   BILI 2020-08-27  Prepared Remarks:\nOperator\nGood day, and wel...
1    GFF 2020-07-30  Prepared Remarks:\nOperator\nThank you for sta...
2   LRCX 2019-10-23  Prepared Remarks:\nOperator\nGood day and welc...
3   BBSI 2019-11-06  Prepared Remarks:\nOperator\nGood day, everyon...
4   CSTE 2019-08-07  Prepared Remarks:\nOperator\nGreetings and wel...


(18013, 3)

In [44]:
save_path = "../data/processed/clean_transcripts.pkl"
clean_data.to_pickle(save_path)

print("Clean dataset saved")


Clean dataset saved
